# 🔀 Multiple providers & models — one unified API

pi-ai's core promise: **the same `Context` runs against any registered model** — you
only swap the `Model` object. This notebook registers several models (all your Azure
deployments, plus an optional Anthropic model) and runs the *identical* prompt against
each, comparing latency, tokens, and cost side-by-side.

> **Kernel:** Deno. Run cells top-to-bottom.

## 1. Load env & register Azure OpenAI

`loadEnvUp()` pulls your `AZURE_PI_TEST_*` keys from a `.env` outside the repo, and
`registerAzure()` registers the Azure OpenAI provider (see `azure.ts`). We keep the
whole `deployments` list this time — every deployment is a model we can compare.

In [ ]:
import { loadEnvUp } from "playground/env";
const env = await loadEnvUp();

In [ ]:
import { registerAzure } from "playground/azure";
const { models, deployments } = registerAzure();

## 2. Build the list of models to compare

A `Models` collection can hold many providers. `models.getModel(provider, id)` resolves
any registered model into the same `Model` shape — that uniformity is the whole point.
Here we resolve one `Model` per Azure deployment.

In [ ]:
import type { Context, Model } from "@earendil-works/pi-ai";

// Every Azure deployment is a model we can compare. getModel() returns undefined
// for unknown ids, so we filter those out.
const compareModels: Model<any>[] = deployments
  .map((id) => models.getModel("azure-openai", id))
  .filter((m): m is Model<any> => m !== undefined);

console.log(`${compareModels.length} model(s) to compare: ${compareModels.map((m) => m.name).join(", ")}`);

## 3. (Optional) Add a second provider — Anthropic

pi-ai ships **built-in providers**. Registering one is a single `setProvider()` call and
it authenticates from the standard env var (`ANTHROPIC_API_KEY`) automatically — no
manual `createProvider()` boilerplate like our custom Azure setup needed.

This cell is **env-guarded**: it only adds Anthropic if you have a key set, so the
notebook still runs fine with Azure alone.

In [ ]:
// OPTIONAL: add Anthropic as a second provider. Runs only if ANTHROPIC_API_KEY is set.
import { anthropicProvider } from "@earendil-works/pi-ai/providers/anthropic";

if (Deno.env.get("ANTHROPIC_API_KEY")) {
  models.setProvider(anthropicProvider());
  const claude = models.getModel("anthropic", "claude-haiku-4-5");
  if (claude) {
    compareModels.push(claude);
    console.log(`✅ Added Anthropic model: ${claude.name}`);
  } else {
    console.log("Anthropic registered, but 'claude-haiku-4-5' isn't in the catalog — skipping.");
  }
} else {
  console.log("No ANTHROPIC_API_KEY — comparing Azure deployments only. (That's fine!)");
}

## 4. Run the same prompt against every model

Notice the loop body is **model-agnostic**: we build one `Context` and call
`models.completeSimple(m, ctx)` — the exact same code regardless of which provider `m`
belongs to. Each call is timed with a monotonic clock and wrapped in `try/catch` so one
failing model never aborts the whole sweep.

In [ ]:
const prompt = "In one sentence, what is the capital of Australia, and one fun fact about it?";

interface Row { model: string; ms: number; inTok: number; outTok: number; cost: number; answer: string; }
const rows: Row[] = [];

for (const m of compareModels) {
  const ctx: Context = {
    systemPrompt: "You are concise. Answer in a single sentence.",
    messages: [{ role: "user", content: prompt, timestamp: Date.now() }],
  };
  const t0 = performance.now();
  try {
    const reply = await models.completeSimple(m, ctx);
    const ms = performance.now() - t0;
    const text = reply.content
      .filter((b) => b.type === "text")
      .map((b) => (b as { text: string }).text)
      .join(" ")
      .replace(/\s+/g, " ")
      .trim();
    rows.push({
      model: m.name,
      ms: Math.round(ms),
      inTok: reply.usage.input,
      outTok: reply.usage.output,
      cost: reply.usage.cost.total,
      answer: text.slice(0, 90),
    });
  } catch (e) {
    rows.push({
      model: m.name,
      ms: Math.round(performance.now() - t0),
      inTok: 0,
      outTok: 0,
      cost: 0,
      answer: `⚠️ ${String(e).replace(/\s+/g, " ").slice(0, 90)}`,
    });
  }
}

## 5. Compare the results

`console.table` renders an aligned grid. Columns: latency (ms), input/output tokens,
cost (USD — `0` for our Azure deployments since we left their cost table at zero), and a
snippet of each answer.

In [ ]:
console.table(
  rows.map((r) => ({
    Model: r.model,
    ms: r.ms,
    in: r.inTok,
    out: r.outTok,
    "$": r.cost.toFixed(6),
    Answer: r.answer,
  })),
);

console.log(`\nCompared ${rows.length} model(s) on the same prompt.`);

## ✅ What just happened

1. `registerAzure()` gave us a `Models` collection with every Azure deployment.
2. We resolved each deployment into a uniform `Model` via `getModel()`.
3. Optionally registered **Anthropic** with a one-line `setProvider(anthropicProvider())`
   — built-in providers need no manual wiring.
4. Ran the **identical** `Context` + `completeSimple` loop across every model and compared
   latency, tokens, and cost.

**The takeaway:** switching models or providers is just swapping the `Model` object.
Your `Context`, tools, streaming loop, and agent code stay *exactly the same*. That
provider-independence is what pi-ai buys you.

Next: **10 — The agent framework** (`@earendil-works/pi-agent-core`), which wraps the
tool-calling loop into a reusable `Agent`.